## CONFIG

In [9]:
import re, unicodedata, difflib

## List of true labels in Italian
true_labels = [
    "arrivare","andare","aspettare","avere","capire","chiamare","chiedere","conoscere","dare","dire","dovere",
    "essere","fare","mettere","potere","prendere","sapere","sentire","trovare","venire","aprire","chiudere",
    "mangiare","bere","accendere","spegnere","volere","bene","si","no","più","poco","molto","sempre","adesso",
    "poi","male","sopra","sotto","destra","sinistra","avanti","indietro","oggi","domani","ieri","forse","prima",
    "perchè","anche","come","però","quindi","quando","dove","se","oppure","io","lui","lei","noi","voi","loro",
    "tu","questo","quello","buono","bello","cattivo","brutto","grande","piccolo","nuovo","vecchio","cosa","parte",
    "anno","casa","problema","aiuto","tempo","lavoro","persona","acqua","cibo","bisogno","donna","uomo","gruppo",
    "guerra","idea","macchina","mano","oggetto","telefono","computer","domanda","uno","mille","paura","ansia",
    "gioia","felicità","tristezza","serenità","amore","morte","bagno","dolore","riposo"
]

def _strip_diacritics(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def _robust_decode(x) -> str:
    if isinstance(x, (bytes, bytearray)):
        for enc in ('utf-8', 'latin-1', 'cp1252'):
            try:
                s = x.decode(enc)
                break
            except Exception:
                continue
    else:
        s = str(x)

    s = s.replace('\x00', '').strip()

    if ('Ã' in s) or ('Â' in s):
        try:
            s = s.encode('latin-1', 'ignore').decode('utf-8', 'ignore')
        except Exception:
            pass

    s = unicodedata.normalize('NFC', s)
    return s

def canonicalize_label(raw) -> str:
    s = _robust_decode(raw).lower()

    s = re.sub(r'\s+', ' ', s).strip()
    s = re.sub(r'_img$', '', s)

    s = (s.replace('’', "'")
           .replace('‘', "'")
           .replace('“', '"')
           .replace('”', '"'))
    s = s.replace("''", "'").replace("´", "'").replace("`", "'").strip()

    fixes = {
        "perche": "perchè", "perche'": "perchè", "perch'e": "perchè", "perchà''": "perchè",
        "pero": "però", "piu": "più", "serenita": "serenità", "felicita": "felicità"
    }
    if s in fixes: 
        return fixes[s]

    if s in true_labels:
        return s

    s_ascii = _strip_diacritics(s)
    candidates_ascii = [_strip_diacritics(t) for t in true_labels]

    match1 = difflib.get_close_matches(s_ascii, candidates_ascii, n=1, cutoff=0.7)
    if match1:
        idx = candidates_ascii.index(match1[0])
        return true_labels[idx]

    match2 = difflib.get_close_matches(s, true_labels, n=1, cutoff=0.6)
    if match2:
        return match2[0]

    return s


In [ ]:
import h5py
import pandas as pd
from pathlib import Path
import json

def build_eeg_dataframe(h5_dir, label_map_path):
    with open(label_map_path, "r", encoding="utf-8") as f:
        label2idx = json.load(f)

    rows = []
    for file in sorted(Path(h5_dir).glob("*.h5")):
        subject_id, session_id = file.stem.split("_")
        with h5py.File(file, "r") as f:
            data = f["data"]
            labels = f["labels"][:] # type: ignore
            n_epochs, n_channels, n_samples = data.shape # type: ignore

            for i, lbl_raw in enumerate(labels): # type: ignore
                label_name = canonicalize_label(lbl_raw)
                label_idx = label2idx.get(label_name, -1)
                rows.append({
                    "subject_id": subject_id,
                    "session_id": session_id,
                    "epoch_idx": i,
                    "label_name": label_name,
                    "label_idx": label_idx,
                    "n_channels": n_channels,
                    "n_samples": n_samples,
                    "fs": 256,
                    "path_h5": str(file)
                })

    df = pd.DataFrame(rows)
    return df


## 🧠 EEG Intelligent DataFrame – Overview

This DataFrame acts as the **central index** for your entire EEG dataset.  
Instead of duplicating raw data, it keeps **structured metadata** that describes every EEG epoch — including where it’s stored, which subject/session it belongs to, and its semantic label.

---

### 📁 Structure

Each row of the DataFrame corresponds to **one EEG epoch** and contains:

| Column | Description |
|:--------|:-------------|
| `subject_id` | ID of the participant (e.g. `11`) |
| `session_id` | Recording session (e.g. `S002`) |
| `epoch_idx` | Index of the epoch inside the `.h5` file |
| `label_name` | Semantic label (e.g. *"felicità"*, *"paura"*) |
| `label_idx` | Numerical class index from `label2idx.json` |
| `n_channels` | Number of EEG channels in that epoch |
| `n_samples` | Number of samples per channel |
| `path_h5` | Absolute path to the `.h5` file containing the signal |

---

### ⚙️ Purpose

The **EEG Intelligent DataFrame** serves as a lightweight, queryable "map" of your dataset.  
It allows you to:

1. **Explore** dataset composition and class balance.  
2. **Filter and retrieve** EEG signals on demand (by subject, session, or label).  
3. **Attach new features** (e.g., spectral power, connectivity metrics).  
4. **Build higher-level datasets** for PyTorch or PyTorch Geometric (graphs).  
5. **Visualize** or debug specific epochs without reloading everything.



In [11]:
from pathlib import Path

project_root = Path.cwd().parents[0] 

h5_dir = project_root / "data" / "processed"
label_map_path = project_root / "data" / "interim" / "label2idx.json"
output_csv = project_root / "data" / "interim" / "eeg_metadata.csv"

meta_df_eeg = build_eeg_dataframe(
    h5_dir=str(h5_dir),
    label_map_path=str(label_map_path)
)

meta_df_eeg.to_csv(output_csv, index=False)

meta_df_eeg

,subject_id,session_id,epoch_idx,label_name,label_idx,n_channels,n_samples,fs,path_h5
0,11,S001,0,cibo,84,61,384,256,/Users/dani/Documents/Thesis_ImaSpe/Daniele_IS...
1,11,S001,1,aprire,20,61,384,256,/Users/dani/Documents/Thesis_ImaSpe/Daniele_IS...
2,11,S001,2,mettere,13,61,384,256,/Users/dani/Documents/Thesis_ImaSpe/Daniele_IS...
3,11,S001,3,nuovo,72,61,384,256,/Users/dani/Documents/Thesis_ImaSpe/Daniele_IS...
4,11,S001,4,bello,67,61,384,256,/Users/dani/Documents/Thesis_ImaSpe/Daniele_IS...
...,...,...,...,...,...,...,...,...,...
1089,11,S005,215,avere,3,61,384,256,/Users/dani/Documents/Thesis_ImaSpe/Daniele_IS...
1090,11,S005,216,mangiare,22,61,384,256,/Users/dani/Documents/Thesis_ImaSpe/Daniele_IS...
1091,11,S005,217,cosa,74,61,384,256,/Users/dani/Documents/Thesis_ImaSpe/Daniele_IS...
1092,11,S005,218,nuovo,72,61,384,256,/Users/dani/Documents/Thesis_ImaSpe/Daniele_IS...


In [27]:
import pandas as pd
# Number of occurrences of each label
print(meta_df_eeg["label_name"].value_counts().head(10))


label_name
cibo        10
acqua       10
spegnere    10
dolore      10
trovare     10
sempre      10
avere       10
prendere    10
quando      10
potere      10
Name: count, dtype: int64


In [28]:
# Number of epochs per subject
print(meta_df_eeg.groupby("subject_id")["epoch_idx"].count())


subject_id
11    1094
Name: epoch_idx, dtype: int64


In [29]:
# Classes present in each subject
print(meta_df_eeg.groupby("subject_id")["label_name"].nunique())


subject_id
11    110
Name: label_name, dtype: int64


In [30]:
# Percentage of each class
print(meta_df_eeg["label_name"].value_counts(normalize=True) * 100)


label_name
cibo        0.914077
acqua       0.914077
spegnere    0.914077
dolore      0.914077
trovare     0.914077
              ...   
se          0.914077
prima       0.914077
uno         0.731261
si          0.731261
telefono    0.731261
Name: proportion, Length: 110, dtype: float64


In [22]:
# Each epoch of subject 11 with class "felicità"
subset = meta_df_eeg.query("subject_id == '11' and label_name == 'felicità'")
print(subset)

     subject_id session_id  epoch_idx label_name  label_idx  n_channels  \
105          11       S001        105   felicità        102          61   
215          11       S001        215   felicità        102          61   
246          11       S002         26   felicità        102          61   
353          11       S002        133   felicità        102          61   
530          11       S003         96   felicità        102          61   
640          11       S003        206   felicità        102          61   
750          11       S004         96   felicità        102          61   
860          11       S004        206   felicità        102          61   
949          11       S005         75   felicità        102          61   
1059         11       S005        185   felicità        102          61   

      n_samples   fs                                            path_h5  
105         384  256  /Users/dani/Documents/Thesis_ImaSpe/Daniele_IS...  
215         384  256  /Use

## EEG Spectral Feature Extraction

For each EEG epoch (1.5 s, 256 Hz sampling rate), spectral features were extracted using **Welch’s Power Spectral Density (PSD)** method.  
The PSD represents how signal power is distributed across frequencies, measured in **µV²/Hz**, since the EEG amplitude was originally expressed in microvolts.

The PSD was integrated within canonical EEG frequency bands to obtain **band power values** in **µV²**:

| Band | Frequency Range (Hz) | Description |
|:-----|:---------------------|:-------------|
| **Delta (δ)** | 1 – 4 | Slow-wave activity, associated with deep sleep or low vigilance |
| **Theta (θ)** | 4 – 8 | Memory processes, drowsiness, limbic activation |
| **Alpha (α)** | 8 – 13 | Relaxed wakefulness, visual idling, eyes-closed resting state |
| **Beta (β)** | 13 – 30 | Motor activity, alertness, active cognitive processing |
| **Gamma (γ)** | 30 – 45 | Fast oscillations, sensory binding, high-level cognition |

### Computed Metrics

Each epoch is described by the following metrics:

| Metric | Definition | Unit | Description |
|:--------|:------------|:------|:-------------|
| **`total_power`** | \(\displaystyle P_\text{tot} = \int_{1}^{45} PSD(f)\,df\) | µV² | Total signal power across 1–45 Hz |
| **`delta`, `theta`, `alpha`, `beta`, `gamma`** | Band power from integration within each band | µV² | Absolute power per band |
| **`*_rel`** | \(\displaystyle P_\text{band} / P_\text{tot}\) | dimensionless | Relative contribution of each band to total power |
| **`alpha_beta_ratio`** | \(\displaystyle \frac{P_\alpha}{P_\beta}\) | dimensionless | Indicator of relaxation vs. activation |
| **`theta_alpha_ratio`** | \(\displaystyle \frac{P_\theta}{P_\alpha}\) | dimensionless | Cognitive fatigue or attentional engagement index |

The relative power values (`*_rel`) sum approximately to 1, representing the normalized spectral composition of each epoch.

### Physiological Interpretation

- **High delta/theta** → low vigilance or drowsy states  
- **High alpha** → relaxed or eyes-closed resting condition  
- **High beta** → active engagement or motor planning  
- **High gamma** → fast cognitive or perceptual integration  
- **Alpha/Beta ratio** → higher in calm or relaxed conditions  
- **Theta/Alpha ratio** → higher in fatigue or stress

---


In [26]:
df_band=pd.read_csv("/Users/dani/Documents/Thesis_ImaSpe/Daniele_IS_Thesis/data/interim/bandpowers.csv")   

df_band.tail()

,subject_id,session_id,epoch_idx,channel,label_name,total_power,delta,theta,alpha,beta,gamma,delta_rel,theta_rel,alpha_rel,beta_rel,gamma_rel,alpha_beta_ratio,theta_alpha_ratio
66729,[11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 ...,S005,219,EEG57,quando_img,13.624916,3.910867,2.352862,2.706755,3.466010,0.978630,0.287038,0.172688,0.198662,0.254388,0.071826,0.780943,0.869256
66730,[11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 ...,S005,219,EEG58,quando_img,4.856166,0.686602,1.837064,1.045630,1.077657,0.189409,0.141388,0.378295,0.215320,0.221915,0.039004,0.970281,1.756897
66731,[11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 ...,S005,219,EEG59,quando_img,5.204909,0.927647,2.238799,1.164351,0.664101,0.204067,0.178225,0.430132,0.223702,0.127591,0.039207,1.753274,1.922788
66732,[11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 ...,S005,219,EEG60,quando_img,9.790245,2.832883,3.479737,1.199867,1.549606,0.673079,0.289358,0.355429,0.122557,0.158281,0.068750,0.774304,2.900103
66733,[11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 ...,S005,219,EEG61,quando_img,11.965551,3.204207,5.279711,2.097334,1.148027,0.229288,0.267786,0.441243,0.175281,0.095944,0.019162,1.826903,2.517344
